In [ ]:

import gzip, json, struct, urllib.request
from multiprocessing import Pool
B = "https://storage.googleapis.com/gresearch/android_control/"
splits = json.load(urllib.request.urlopen(B + "splits.json"))
subs = json.load(urllib.request.urlopen(B + "test_subsplits.json"))
TEST = set(splits["test"]); TRAIN = set(splits["train"])
SUB = {e: k for k, v in subs.items() for e in v}

def varint(b, i):
    r = s = 0
    while True:
        x = b[i]; i += 1; r |= (x & 0x7f) << s; s += 7
        if x < 128: return r, i

def fields(b):
    i = 0; out = []
    while i < len(b):
        k, i = varint(b, i); f, t = k >> 3, k & 7
        if t == 0: v, i = varint(b, i)
        elif t == 2: n, i = varint(b, i); v = b[i:i + n]; i += n
        elif t == 5: v = b[i:i + 4]; i += 4
        elif t == 1: v = b[i:i + 8]; i += 8
        else: raise ValueError(t)
        out.append((f, t, v))
    return out

def example(rec, skip=("screenshots",)):
    feats = {}
    for _, _, v in fields(rec):
        for _, _, entry in fields(v):
            kv = dict((a, c) for a, _, c in fields(entry))
            key = kv[1].decode()
            if key in skip: continue
            f, _, val = fields(kv[2])[0]
            if f == 1: feats[key] = [x for _, _, x in fields(val)]
            elif f == 3:
                vals = []; 
                for ff, tt, x in fields(val):
                    if tt == 2:
                        j = 0
                        while j < len(x): n, j = varint(x, j); vals.append(n)
                    else: vals.append(x)
                feats[key] = vals
    return feats

S = lambda b: b.decode("utf-8", "replace") if isinstance(b, bytes) else ""
def nodes(forest):
    out = []
    for _, _, wv in fields(forest):
        w = dict((a, c) for a, _, c in fields(wv))
        if 11 not in w: continue
        for a, _, n in fields(w[11]):
            if a != 1: continue
            d = {}
            for f, t, v in fields(n): d.setdefault(f, v)
            r = dict((a2, c2) for a2, _, c2 in fields(d.get(2, b"")))
            e = {"id": d.get(1, 0), "cls": S(d.get(3)).split(".")[-1], "desc": S(d.get(4)), "hint": S(d.get(5)),
                 "pkg": S(d.get(6)), "text": S(d.get(7)), "rid": S(d.get(10)).split("/")[-1],
                 "b": [r.get(1, 0), r.get(2, 0), r.get(3, 0), r.get(4, 0)],
                 "click": d.get(14, 0), "edit": d.get(15, 0), "focus": d.get(18, 0), "long": d.get(19, 0),
                 "scroll": d.get(21, 0), "check": d.get(12, 0), "checked": d.get(13, 0), "sel": d.get(22, 0),
                 "vis": d.get(23, 0)}
            if e["vis"] and (e["text"] or e["desc"] or e["hint"] or e["rid"] or e["click"] or e["edit"] or e["scroll"]):
                out.append(e)
    return out

def shard(k):
    url = B + f"android_control-{k:05d}-of-00020"
    rows = []; ntr = 0; bad = 0
    with gzip.GzipFile(fileobj=urllib.request.urlopen(url, timeout=600)) as g:
        while True:
            h = g.read(12)
            if len(h) < 12: break
            n = struct.unpack("<Q", h[:8])[0]; rec = g.read(n); g.read(4)
            try:
                e = example(rec); ep = e["episode_id"][0]
                if "step_instructions" not in e or "actions" not in e: bad += 1; continue
            except Exception:
                bad += 1; continue
            split = "test" if ep in TEST else "train" if ep in TRAIN else "val"
            if split == "val" or (split == "train" and ntr >= 100000): continue
            ntr += split == "train"
            trees = e["accessibility_trees"]; W = e["screenshot_widths"]; H = e["screenshot_heights"]
            for i, (ins, act) in enumerate(zip(e["step_instructions"], e["actions"])):
                rows.append({"ep": ep, "split": split, "sub": SUB.get(ep), "step": i, "goal": S(e["goal"][0]),
                             "instr": S(ins), "action": json.loads(act), "w": W[i], "h": H[i], "els": nodes(trees[i])})
    print(k, "shard done, rows", len(rows), "skipped", bad, flush=True)
    return rows

with Pool(4) as p:
    total = {"test": 0, "train": 0}
    with gzip.open("/kaggle/working/androidcontrol_text.jsonl.gz", "wt") as out:
        for rows in p.imap_unordered(shard, range(20)):
            for r in rows:
                out.write(json.dumps(r) + "\n"); total[r["split"]] += 1
            print(total, flush=True)
